# KrishiMitra — Plant disease detection experiments (Phases 12–14)

Run this notebook on **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU).

What it does, in order:
1. Gets the code (your `ml/` folder on Google Drive) and the PlantVillage dataset (Kaggle).
2. Cleans the data and saves a fixed stratified train/val/test split.
3. Explores the dataset (class distribution, image statistics, sample grid).
4. Trains the baseline CNN and three transfer-learning models.
5. Evaluates every model on the **held-out test set** and on **PlantDoc field photos**.
6. Builds the comparison table and exports the chosen model to ONNX (verified against PyTorch).

All numbers in your report must come from the files this notebook writes in `results/`.

In [ ]:
# 1. Mount Google Drive and go to the project's ml/disease_detection folder
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/krishimitra/ml/disease_detection
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the few packages Colab does not ship with
!pip -q install onnx onnxruntime tabulate kaggle

### Download PlantVillage from Kaggle
Upload your `kaggle.json` (Kaggle → Settings → Create New Token) when asked. The dataset is about 2 GB.

In [ ]:
from google.colab import files
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()  # choose kaggle.json
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
# Copy to the Colab disk (faster than reading thousands of files from Drive)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data --unzip -q
!ls "/content/data/plantvillage dataset" 

In [ ]:
# 3. Clean + split (seed 42). Saved to splits/ so every model uses the same images.
!python src/prepare_split.py --data-dir "/content/data/plantvillage dataset/color" --out-dir splits
!python src/eda.py --split splits/train.csv --out results/eda

In [ ]:
from IPython.display import Image, display
display(Image('results/eda/class_distribution.png', width=700))

### Optional: hyperparameter search (validation set only)
Takes about an hour on a T4. Copy the best values into the config before full training.

In [ ]:
# !python src/tune.py --config configs/efficientnet_b0.yaml --epochs 3

In [ ]:
# 4. Train all four models (about 15–60 minutes each on a T4)
for cfg in ['baseline_cnn', 'mobilenet_v2', 'mobilenet_v3_large', 'efficientnet_b0']:
    !python src/train.py --config configs/{cfg}.yaml

In [ ]:
# 5. Evaluate every model on the held-out test set
for run in ['baseline_cnn', 'mobilenet_v2', 'mobilenet_v3_large', 'efficientnet_b0']:
    !python src/evaluate.py --run results/{run} --split splits/test.csv --tag test
!python src/compare.py --tag test

### Independent field test (PlantDoc)
PlantVillage images have plain backgrounds; PlantDoc has real field photos. Expect a large drop — report it honestly.

In [ ]:
!git clone -q https://github.com/pratikkayal/PlantDoc-Dataset /content/PlantDoc
!python src/prepare_external.py --plantdoc-dir /content/PlantDoc/test --out splits/plantdoc_test.csv
for run in ['baseline_cnn', 'mobilenet_v2', 'mobilenet_v3_large', 'efficientnet_b0']:
    !python src/evaluate.py --run results/{run} --split splits/plantdoc_test.csv --tag field
!python src/compare.py --tag field

In [ ]:
display(Image('results/comparison_test.png', width=600))
display(Image('results/efficientnet_b0/curves.png', width=800))
display(Image('results/efficientnet_b0/confusion_matrix_test.png', width=800))

In [ ]:
# 6. Export the chosen model (change the run name if another model wins on your results)
CHOSEN = 'efficientnet_b0'
!python src/export_onnx.py --run results/{CHOSEN} --out ../../trained_models --check-csv splits/test.csv
# Download trained_models/disease_model.onnx and disease_labels.json into your project's trained_models/ folder.